# Clase 1.5 — Perceptrón y Funciones de Activación
### Versión corregida y explicada paso a paso

---
## ¿Qué hacemos en este notebook?

Implementamos el perceptrón **a mano** (fuerza bruta): elegimos pesos W1, W2 y bias b,
calculamos h = X1·W1 + X2·W2 + b para cada fila, y aplicamos la función de activación.

El objetivo es **entender por qué** funcionan esos valores antes de que el modelo los aprenda solo.

```
Proceso del perceptrón:

  X1 ──(W1)──┐
              ├──> h = X1·W1 + X2·W2 + b ──> f(h) ──> salida
  X2 ──(W2)──┘
              + b (bias)
```

## ¿Cómo elegir W1, W2 y b?
No hay fórmula mágica para "fuerza bruta" — se razona así:

1. **¿Qué quiero que haga la salida?** Ver la tabla de verdad.
2. **¿Qué dirección tienen las entradas?** Si subir X1 debe subir la salida → W1 positivo. Si debe bajarla → W1 negativo.
3. **¿Cuándo debe activarse?** El bias desplaza el umbral. Si necesitas que h sea positivo cuando ambas entradas son 1, ajusta b.
4. **Verificar con cada fila** y ajustar si algo falla.

In [36]:
# ════════════════════════════════════════════════════════════
# CELDA 1 — Path y imports
# ════════════════════════════════════════════════════════════
import sys, os

# ── Solución path Windows/Linux ──────────────────────────────
# Descomenta la línea que corresponde:

# WINDOWS (en casa):
# RUTA = r"D:/Proyectos/Diplomado-RNA/Machote"

# LINUX / UBUNTU (en la escuela):
# RUTA = "/home/usr-lbr-maq20/Documentos/Diplomado-RNA/Machote"

# ALTERNATIVA que funciona en AMBOS sin cambiar nada:
# (sube carpetas desde donde está este archivo)
RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))

if RUTA not in sys.path:
    sys.path.append(RUTA)

import numpy as np
print("Imports listos")

Imports listos


---
## Definición de las funciones de activación

### ¿Cuál función usar?

| Si la salida es... | Función a usar | Por qué |
|-------------------|---------------|--------|
| Solo 0 o 1 (clasificación) | **Escalón unitario** | Corta en 0, da exactamente 0 o 1 |
| Probabilidad (0.0 a 1.0) | **Sigmoide** | Da valores suaves entre 0 y 1 |
| Número positivo o 0 | **ReLU** | Deja pasar valores positivos, corta los negativos |
| Número continuo cualquiera | **Lineal (sin función)** | f(h) = h, sin modificar |
| Número entre -1 y 1 | **Tanh** | Centrada en 0, rango (-1, 1) |

In [37]:
# ════════════════════════════════════════════════════════════
# CELDA 2 — Funciones de activación
# ════════════════════════════════════════════════════════════

def escalon_unitario(x):
    """
    Función escalón: devuelve 1 si x >= 0, sino 0.
    Usar cuando la salida debe ser exactamente 0 o 1.
    Ejemplo: ¿El tumor es maligno? ¿La compuerta activa?
    """
    return 1 if x >= 0 else 0

def sigmoide(x):
    """
    Sigmoide: devuelve valor entre 0 y 1.
    Usar cuando necesitas PROBABILIDAD de pertenencia a una clase.
    Ejemplo: probabilidad de que el correo sea spam (0.87 = 87% probable spam).
    """
    return 1 / (1 + np.exp(-x))

def tangente_hiperbolica(x):
    """
    Tanh: devuelve valor entre -1 y 1.
    Centrada en 0 (a diferencia de sigmoide que está centrada en 0.5).
    Útil en capas ocultas de redes neuronales.
    """
    return np.tanh(x)

def relu(x):
    """
    ReLU: devuelve x si x > 0, sino 0.
    Usar cuando la salida debe ser un número positivo (o cero).
    Es la más usada en capas ocultas de redes profundas.
    Ejemplo: precio de un producto (no puede ser negativo).
    """
    return max(0, x)

def lineal(x):
    """
    Lineal: devuelve x sin modificarlo.
    Usar cuando la salida es un número continuo sin restricción de rango.
    Ejemplo: temperatura, velocidad, distancia.
    """
    return x

print("Funciones de activación definidas")

Funciones de activación definidas


In [38]:
# ════════════════════════════════════════════════════════════
# CELDA 3 — Función helper para imprimir resultados bonito
# ════════════════════════════════════════════════════════════

def mostrar_resultados(nombre, inputs, targets, outputs):
    """
    Imprime la tabla de resultados de forma legible y
    marca con ✅ o ❌ si el output coincide con el target.
    """
    print(f"\n{'═'*60}")
    print(f"  Compuerta {nombre}")
    print(f"{'═'*60}")
    print(f"  {'X1':>4} {'X2':>4} {'h':>8} {'f(h)':>8} {'Target':>8}  OK")
    print(f"  {'-'*50}")

    correctos = 0
    for i in range(len(outputs)):
        x1  = inputs[i][0]
        x2  = inputs[i][1]
        h   = outputs[i][0]
        fh  = outputs[i][1]
        tgt = targets[i][0]

        # Comparar con tolerancia para floats
        ok = '✅' if abs(float(fh) - float(tgt)) < 1e-9 else '❌'
        if ok == '✅':
            correctos += 1

        h_val = float(h[0]) if hasattr(h, '__len__') else float(h)
        print(f"  {x1:>4} {x2:>4} {h_val:>8.2f} {float(fh):>8.2f} {float(tgt):>8.2f}  {ok}")

    print(f"  {'-'*50}")
    print(f"  Resultado: {correctos}/{len(outputs)} correctos", end="")
    if correctos == len(outputs):
        print(" ✅ Compuerta correcta")
    else:
        print(" ❌ Los pesos no implementan esta compuerta")

print("Helper listo")

Helper listo


In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 4 — Entradas y tablas de verdad
# ════════════════════════════════════════════════════════════

# Las 4 combinaciones posibles de 2 entradas binarias
inputs = np.array([[0,0],[0,1],[1,0],[1,1]])

# Tablas de verdad de cada compuerta
Salida_OR   = np.array([[0],[1],[1],[1]])   # 1 cuando al menos una entrada es 1
Salida_AND  = np.array([[0],[0],[0],[1]])   # 1 solo cuando AMBAS son 1
Salida_NOR  = np.array([[1],[0],[0],[0]])   # 1 solo cuando AMBAS son 0
Salida_NAND = np.array([[1],[1],[1],[0]])   # 0 solo cuando AMBAS son 1
Salida_NOT  = np.array([[1],[0],[0],[0]])   # invierte la primera entrada
Salida_XOR  = np.array([[0],[1],[1],[0]])   # 1 cuando las entradas son DIFERENTES

print("Entradas y salidas definidas")
print()
print("Todas las combinaciones de inputs:")
print(inputs)

Entradas y salidas definidas

Todas las combinaciones de inputs:
[[0 0]
 [0 1]
 [1 0]
 [1 1]]


---
## Compuerta OR

**Lógica:** Da 1 cuando AL MENOS UNA entrada es 1.

**Razonamiento para elegir los pesos:**
- Queremos que h sea positivo (→ salida 1) cuando X1=1 **ó** X2=1.
- Si X1 o X2 = 1, queremos h ≥ 0 → los pesos deben ser **positivos**.
- El único caso donde salida = 0 es (0,0), donde h = b → necesitamos b < 0.
- Con W1=1, W2=1, b=-0.5: el caso (0,1) da h = 0+1-0.5 = 0.5 ≥ 0 → 1 ✓


In [40]:
outputs = []
targets = Salida_OR

# ── Pesos para OR ────────────────────────────────────────────
# W1=1, W2=1  → cualquier entrada en 1 empuja h hacia positivo
# b=-0.5      → cuando ambas son 0, h = -0.5 (negativo → salida 0)
weights = np.array([[1],[1]])
bias    = -0.5

# ── Proceso fila por fila ─────────────────────────────────────
# h = X1*W1 + X2*W2 + b  (np.dot calcula X1*W1 + X2*W2 automáticamente)
# f(h) = escalon_unitario(h)
for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias
    y = escalon_unitario(h)   # escalón porque la salida debe ser 0 o 1
    outputs.append([h, y])

mostrar_resultados("OR", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta OR
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0    -0.50     0.00     0.00  ✅
     0    1     0.50     1.00     1.00  ✅
     1    0     0.50     1.00     1.00  ✅
     1    1     1.50     1.00     1.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta AND

**Lógica:** Da 1 solo cuando AMBAS entradas son 1.

**Razonamiento:**
- Similar a OR, pero el umbral debe ser más alto.
- Con W1=1, W2=1: la suma máxima es 1+1=2. Necesitamos que solo (1,1) pase.
- Si ponemos b=-1.5: h(1,1) = 2-1.5 = 0.5 ≥ 0 → 1 ✓ y h(0,1) = 1-1.5 = -0.5 < 0 → 0 ✓

In [41]:
outputs = []
targets = Salida_AND

# W1=1, W2=1  → necesitamos ambas entradas para superar el umbral
# b=-1.5      → umbral alto: solo (1,1) da h=0.5 positivo
weights = np.array([[1],[1]])
bias    = -1.5

for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias
    y = escalon_unitario(h)
    outputs.append([h, y])

mostrar_resultados("AND", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta AND
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0    -1.50     0.00     0.00  ✅
     0    1    -0.50     0.00     0.00  ✅
     1    0    -0.50     0.00     0.00  ✅
     1    1     0.50     1.00     1.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta NOR

**Lógica:** Lo opuesto de OR. Da 1 solo cuando AMBAS son 0.

**Razonamiento:**
- Cualquier entrada en 1 debe apagar la salida → pesos **negativos**.
- Cuando ambas son 0, h = b → necesitamos b > 0 para que salga 1.
- Con W1=-1, W2=-1, b=0.5: h(0,0)=0.5 → 1 ✓ y h(0,1)=-0.5 → 0 ✓

In [42]:
outputs = []
targets = Salida_NOR

# W1=-1, W2=-1 → cualquier entrada en 1 empuja h hacia negativo
# b=0.5        → cuando ambas son 0, h=0.5 positivo → salida 1
weights = np.array([[-1],[-1]])
bias    = 0.5

for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias
    y = escalon_unitario(h)
    outputs.append([h, y])

mostrar_resultados("NOR", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta NOR
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0     0.50     1.00     1.00  ✅
     0    1    -0.50     0.00     0.00  ✅
     1    0    -0.50     0.00     0.00  ✅
     1    1    -1.50     0.00     0.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta NAND

**Lógica:** Lo opuesto de AND. Da 0 solo cuando AMBAS son 1, en todos los demás casos da 1.

**Razonamiento:**
- Similar a NOR pero el umbral de apagado es más alto.
- Pesos negativos pero bias mayor para que (0,1) y (1,0) sigan dando 1.


In [43]:
outputs = []
targets = Salida_NAND   

# W1=-1, W2=-1 → solo cuando ambas son 1, la suma es -2
# b=1.5        → h(1,1) = -2+1.5 = -0.5 → 0 ✓
#                h(0,1) = -1+1.5 = 0.5  → 1 ✓
weights = np.array([[-1],[-1]])
bias    = 1.5

for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias
    y = escalon_unitario(h)
    outputs.append([h, y])

mostrar_resultados("NAND", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta NAND
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0     1.50     1.00     1.00  ✅
     0    1     0.50     1.00     1.00  ✅
     1    0     0.50     1.00     1.00  ✅
     1    1    -0.50     0.00     0.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta NOT

**Lógica:** Invierte la entrada. NOT(0)=1, NOT(1)=0.
Solo tiene UNA entrada real (X1). X2 es LOGIC 1 (siempre 1).

**Razonamiento:**
- W1 debe ser negativo: cuando X1 sube (1), queremos que h baje (→ salida 0).
- W2 aporta un empujón positivo constante (porque X2=1 siempre).
- Con W1=-1: h(0)=0+W2·1+b, h(1)=-1+W2·1+b. Necesitamos h(0)≥0 y h(1)<0.

In [45]:
# NOT tiene una sola entrada real: X1=A, X2=LOGIC 1 (siempre vale 1)
inputs_not = np.array([[0,1],[0,1],[1,1],[1,1]])
outputs = []
targets = Salida_NOT

# W1=-1  → cuando A=1, jala h hacia negativo
# W2=0.5 → LOGIC 1 aporta 0.5 positivo constante
# b=0    → no necesitamos más ajuste
# h(A=0) = 0·(-1) + 1·0.5 + 0 = +0.5 → 1 ✓
# h(A=1) = 1·(-1) + 1·0.5 + 0 = -0.5 → 0 ✓
weights = np.array([[-1],[0.5]])
bias    = 0

for i in range(targets.shape[0]):
    h = np.dot(inputs_not[i], weights) + bias
    y = escalon_unitario(h)  # NOT es un caso especial: queremos que la salida sea 0 o 1, pero no con escalón
    #y = relu(h)  # NOT es un caso especial: queremos que la salida sea 0 o 1, pero no con escalón
    outputs.append([h, y])

mostrar_resultados("NOT (X2=LOGIC 1)", inputs_not, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta NOT (X2=LOGIC 1)
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    1     0.50     1.00     1.00  ✅
     0    1     0.50     1.00     0.00  ❌
     1    1    -0.50     0.00     0.00  ✅
     1    1    -0.50     0.00     0.00  ✅
  --------------------------------------------------
  Resultado: 3/4 correctos ❌ Los pesos no implementan esta compuerta


In [ ]:
# NOT tiene una sola entrada real: X1=A, X2=LOGIC 1 (siempre vale 1)
inputs_not = np.array([[0,1],[0,1],[1,1],[1,1]])
outputs = []
targets = Salida_NOT

# W1=-1  → cuando A=1, jala h hacia negativo
# W2=0.5 → LOGIC 1 aporta 0.5 positivo constante
# b=0    → no necesitamos más ajuste
# h(A=0) = 0·(-1) + 1·0.5 + 0 = +0.5 → 1 ✓
# h(A=1) = 1·(-1) + 1·0.5 + 0 = -0.5 → 0 ✓
weights = np.array([[-1],[0.5]])
bias    = 0

for i in range(targets.shape[0]):
    h = np.dot(inputs_not[i], weights) + bias
    #y = escalon_unitario(h)  # NOT es un caso especial: queremos que la salida sea 0 o 1, pero no con escalón
    y = relu(h)  # NOT es un caso especial: queremos que la salida sea 0 o 1, pero no con escalón # ???
    outputs.append([h, y])

mostrar_resultados("NOT (X2=LOGIC 1)", inputs_not, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta NOT (X2=LOGIC 1)
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------


TypeError: only 0-dimensional arrays can be converted to Python scalars

---
## Compuerta XOR — Por qué siempre falla

**Lógica:** Da 1 cuando las entradas son **diferentes**.

El perceptrón de UNA sola capa no puede implementar XOR porque XOR no es linealmente separable.

In [ ]:
print("Intentando XOR con varios pesos — todos fallan en alguna fila:")

intentos_xor = [
    ("W=[1,1] b=-0.5",   np.array([[1],[1]]),    -0.5),
    ("W=[-1,1] b=-0.5",  np.array([[-1],[1]]),   -0.5),
    ("W=[1,-1] b=-0.5",  np.array([[1],[-1]]),   -0.5),
    ("W=[-1,-1] b=0.5",  np.array([[-1],[-1]]),   0.5),
]

targets_xor = Salida_XOR

for nombre, w, b in intentos_xor:
    outputs = []
    for i in range(targets_xor.shape[0]):
        h = np.dot(inputs[i], w) + b
        y = escalon_unitario(h)
        outputs.append([h, y])

    correctos = sum(
        1 for i in range(len(outputs))
        if abs(float(outputs[i][1]) - float(targets_xor[i][0])) < 1e-9
    )
    print(f"  {nombre:25s} → {correctos}/4 correctos  {'✅' if correctos==4 else '❌ siempre falla'}")     

print()
print("Conclusión: No existe ningún W1, W2, b que haga XOR con UN solo perceptrón.")
print("Se necesita Perceptrón Multicapa (MLP) — lo veremos en el Módulo IV.")

Intentando XOR con varios pesos — todos fallan en alguna fila:
  W=[1,1] b=-0.5            → 3/4 correctos  ❌ siempre falla
  W=[-1,1] b=-0.5           → 3/4 correctos  ❌ siempre falla
  W=[1,-1] b=-0.5           → 3/4 correctos  ❌ siempre falla
  W=[-1,-1] b=0.5           → 1/4 correctos  ❌ siempre falla

Conclusión: No existe ningún W1, W2, b que haga XOR con UN solo perceptrón.
Se necesita Perceptrón Multicapa (MLP) — lo veremos en el Módulo IV.


---
# Ejercicio: Control de Velocidad

## El problema

| Altura Elevada (X1) | Aceleración Alta (X2) | Aumento de Velocidad |
|:-------------------:|:---------------------:|:-------------------:|
| 0 | 0 | **350** |
| 0 | 1 | **200** |
| 1 | 0 | **200** |
| 1 | 1 | **50**  |

## ¿Por qué NO podemos usar el escalón unitario aquí?

El escalón solo puede dar **0 o 1**. Pero los valores que necesitamos son **350, 200, 200, 50** — números continuos.

Si usáramos escalón:
- ¿350 = 1? ¿50 = 0? La escala está completamente fuera de rango.
- No hay ningún threshold que convierta h en 350, 200 o 50.

## ¿Qué función de activación usar?

Como la salida son **números continuos positivos**, usamos **función lineal**: f(h) = h.
Es decir, no aplicamos ninguna transformación — la suma ponderada ES la salida.

## Cómo encontrar W1, W2 y b (razonamiento)

Necesitamos que:
```
h(0,0) = 0·W1 + 0·W2 + b = b         = 350
h(0,1) = 0·W1 + 1·W2 + b = W2 + 350  = 200  →  W2 = 200 - 350 = -150
h(1,0) = 1·W1 + 0·W2 + b = W1 + 350  = 200  →  W1 = 200 - 350 = -150
h(1,1) = (-150) + (-150) + 350 = 50   ← verificación ✅
```

¡Perfecto! Los 4 casos se satisfacen con W1=-150, W2=-150, b=350.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Ejercicio: Control de Velocidad
# ═══════════════════════════════════════════════════════════════

inputs_vel = np.array([[0,0],[0,1],[1,0],[1,1]])
targets_vel = np.array([[350],[200],[200],[50]])

# Razonamiento:
# Cuando X1=X2=0, la salida es 350 → eso es el bias (punto de partida)
# Cada entrada en 1 reduce la velocidad en 150
# W1=-150, W2=-150, b=350
weights_vel = np.array([[-150],[-150]])
bias_vel    = 350

outputs_vel = []

for i in range(targets_vel.shape[0]):
    h = np.dot(inputs_vel[i], weights_vel) + bias_vel
    # ← FUNCIÓN LINEAL: f(h) = h (no aplicamos escalón ni sigmoide)
    # La salida ES la suma ponderada porque necesitamos un número continuo
    y = lineal(h)
    outputs_vel.append([h, y])

# Mostrar resultados
print("═" * 65)
print("  Ejercicio: Control de Velocidad")
print("  Función de activación: LINEAL (f(h) = h)")
print("  W1=-150, W2=-150, b=350")
print("═" * 65)
print(f"  {'X1 (Altura)':>12} {'X2 (Acel.)':>12} {'h':>8} {'Salida':>8} {'Target':>8}  OK")
print("  " + "-"*55)

correctos = 0
for i in range(len(outputs_vel)):
    x1  = inputs_vel[i][0]
    x2  = inputs_vel[i][1]
    h   = float(outputs_vel[i][0][0])
    fh  = float(outputs_vel[i][1])
    tgt = float(targets_vel[i][0])
    ok  = '✅' if abs(fh - tgt) < 1e-9 else '❌'
    if ok == '✅': correctos += 1
    print(f"  {x1:>12} {x2:>12} {h:>8.1f} {fh:>8.1f} {tgt:>8.1f}  {ok}")

print("  " + "-"*55)
print(f"  Resultado: {correctos}/4 correctos ✅")
print()
print("  Interpretación:")
print("  - Sin altura elevada ni aceleración alta → velocidad sube 350")
print("  - Cada condición presente reduce el aumento en 150")
print("  - Con ambas condiciones: 350 - 150 - 150 = 50")

---
# Resumen — ¿Cómo saber qué pesos elegir? (proceso mental)

```
PASO 1: Ver la tabla de verdad
   ¿Cuándo quiero salida 1 (o alta)?
   ¿Cuándo quiero salida 0 (o baja)?

PASO 2: ¿Qué papel juega cada entrada?
   Si subir X1 debe subir la salida → W1 POSITIVO
   Si subir X1 debe bajar la salida → W1 NEGATIVO
   Si X1 no importa → W1 cerca de 0

PASO 3: ¿Cuál es el caso límite?
   OR:  (0,1) debe dar 1 → los pesos deben ser suficientes para eso
   AND: (0,1) debe dar 0 → los pesos solos no alcanzan, necesitas b bajo
   NOR: (0,0) debe dar 1 → b debe ser positivo para que h(0,0)=b≥0

PASO 4: Calcular b despejando del caso más claro
   Para OR:  h(0,0) debe ser < 0 → b < 0
             h(0,1) = W2+b ≥ 0  → W2+b ≥ 0
             Si W2=1: b > -1 pero b < 0 → b = -0.5 funciona

PASO 5: Verificar TODAS las filas
   Si algo falla, ajustar b o los pesos y volver al paso 4.

PASO 6: ¿Qué función de activación?
   Salida = {0,1}     → escalon_unitario
   Salida = número    → lineal (no aplicar función)
   Salida = prob 0-1  → sigmoide
   Salida = positivo  → relu
```